# 02b · Git、pre-commit、CI 到底是什么（大白话）

> 这一章是讲解，用我们这两天真实遇到的事情当例子。最后几格是只读的小命令，看一眼我们自己的仓库，不会改任何东西，可以放心运行。

## 一张图看懂整个流程

```
          你的电脑                                         GitHub（网上）
 ┌───────────────────────────────────┐             ┌──────────────────────────────┐
 │ 写代码                              │             │ 仓库：大家共用的那一份          │
 │   ↓                                 │             │   ↓ 每次收到新存档，自动：       │
 │ git add      选这次要存档的改动       │   git push  │ CI：拿一台全新的电脑，          │
 │   ↓                                 │ ──────────▶ │     从零装一遍、测一遍           │
 │ git commit   存档                   │             │   ↓                            │
 │   ↑ 存档前自动：pre-commit 自查（几秒） │             │ ✅ 全绿 / ❌ 有红               │
 └───────────────────────────────────┘             └──────────────────────────────┘
```

一句话：**Git 负责存档，pre-commit 在存档前自查，GitHub 放一份大家共用的，CI 在共用那份上自动复现。**

---

## 1 · Git：代码的存档系统

### 比喻
游戏存档，或者 Word 的"版本历史"，但更强：每一次存档都记下**改了哪些文件、改了什么、谁改的、什么时候、为什么改**。任何时候都能回到以前的任何一个存档。

### 几个词

| 词 | 大白话 | 我们项目里 |
|---|---|---|
| **仓库**（repository） | 一个项目的全部文件，加上全部的存档历史 | `qafi-ml-platform` 这个文件夹。历史存在里面一个隐藏的 `.git` 文件夹里 |
| **提交**（commit） | 一次存档 | 现在有好几次，最后一格可以看到 |
| **暂存**（`git add`） | 选这次存档要包含哪些改动。像寄快递前，先把东西放进箱子 | 我们一般用 `git add -A`：所有改动都放进去 |
| **提交说明**（commit message） | 这次存档做了什么，一句话 | `feat: 03 raw ingestion and 03b public data ingestion; fix CI setup-uv version` |
| **`.gitignore`** | 一张"**不许存档**"的清单 | 数据、密码、Terraform 记账本、虚拟环境都在上面 |

### 为什么提交说明要写清楚
三个月后的你、同事、看简历的面试官，都是靠说明看懂每次存档做了什么。我们遇到过：02 号建仓库那一格被重复运行了两次，每次都存了一档，三次存档的说明一模一样，都是 `platform foundation`，可内容完全不同。推到网上之前，我把后两次合成了一次，说明改成了实际内容。

### 公司里
**所有代码都在 Git 里。没进 Git 的改动，等于不存在**：别人看不到，CI 测不到，电脑坏了就没了。

---

## 2 · GitHub：存档放到网上，大家共用

### 比喻
Git 是你电脑上的存档；GitHub 是网上的"总部"，放着同一个仓库，团队都从这里拿、往这里交。

| 词 | 大白话 |
|---|---|
| **push** | 把你电脑上的新存档，传到 GitHub |
| **pull** | 把 GitHub 上别人的新存档，拉到你电脑 |
| **私有 / 公开** | 我们的仓库现在是私有的，做完放进简历时改成公开 |

### 公司里多两样东西：分支和 Pull Request
- **分支**（branch）：在一份副本上改，不影响主线（`main`）。改坏了也不影响别人
- **Pull Request**（PR）：申请"把我分支上的改动合并进主线"。提交 PR 后，CI 自动检查，同事审查，都通过了才合并

**我们简化了**：只有你一个人，所以直接在 `main` 上提交。其他都和公司一样。

---

## 3 · pre-commit：存档那一刻的自查

### 比喻
出门前检查一下行李：钥匙带了没有，有没有带不该带的东西。

### 它是什么
每次 `git commit`，**在你的电脑上**，自动跑一组检查，几秒钟。有一项不过，这次存档就不会发生。

| 检查 | 拦住什么 |
|---|---|
| ruff | 代码错误、格式不统一 |
| 文件末尾 / 行尾空格 | 无意义的格式差异 |
| check yaml | 写坏了的配置文件 |
| 大文件 | 超过 15 MB 的文件：数据永远不该进 Git |
| 私钥 / AWS 密钥 | 密码和密钥 |
| nbstripout | notebook 的运行输出：里面可能有账号、数据 |

### 为什么要在存档前查
因为**密码一旦进了存档、推到网上，就晚了**：网上有专门扫描密钥的机器人，几分钟内就能发现。在存档那一刻拦住，密码就根本不会离开你的电脑。

### 我们遇到的真实例子
- **它拦住了我自己**：02 号演示"私钥会被拦住"的那一格，代码里写着一句假的私钥开头。第一次提交时，整个 02 号 notebook 都被这个检查拦了下来，最后把那句话拆成两半写才过去。它只认内容，不管你是不是在演示，这正是它可靠的地方
- **第一次提交时 `nbstripout Failed`，提示 `files were modified`**：不是出错，是它把 notebook 整理好了。再 `git add` 一次就能提交

### 放在哪
- 检查的清单：[`.pre-commit-config.yaml`](../../.pre-commit-config.yaml)，在 Git 里，所有人共用同一套
- 装好的"钩子"：`.git/hooks/pre-commit`，**每台电脑要装一次**（02 号第 4 步的 `pre-commit install`）。没装的电脑，提交时就不会自查，所以还需要 CI

---

## 4 · CI：推送之后的自动复现

### 比喻
别的实验室按你的实验记录重做一遍，看能不能得到同样的结果。CI 是每推一次，就自动重做一次。

### 它是什么
每次 push，GitHub 自动拿几台**全新的电脑**（上面没有你电脑里的任何文件），按清单从零装、从零测：

| 检查 | 做什么 |
|---|---|
| lint | ruff 代码检查 |
| test | 全部测试，包括黄金测试：0.480 和 0.914 |
| terraform | 图纸写得对不对（不连 AWS、不建任何东西） |
| docker | 镜像能不能打包 |
| aws | 用临时通行证登录 AWS，问一句"我是谁" |

### 放在哪，在哪看
- 清单：[`.github/workflows/ci.yml`](../../.github/workflows/ci.yml)，在 Git 里
- 结果：GitHub 网页上仓库的 **Actions** 页面。全绿才算过

### 我们遇到的真实例子：CI 红了两次，两次都是本地查不出来的问题
1. **第一次红**：lint 和 test 在"准备环境"这一步就失败了。原因是清单里写的工具版本 `setup-uv@v10` 根本不存在，这个工具只发布 `v10.2.0` 这样的完整版本号。你的电脑上不用这个工具，所以本地什么都没发现。改成 `v10.2.0`，就绿了
2. **第二次红**：aws 检查被 AWS 拒绝。原因是 GitHub 改了临时证明的写法：以前写仓库名，现在在名字后面加上账号和仓库的**永久编号**，这样更安全。AWS 那边的信任条件还是旧写法，所以对不上。修法是把 AWS 的信任条件改成新写法

**这正是 CI 的价值**：问题在一台干净的电脑上暴露出来，而不是等别人用的时候才发现。

### pre-commit 和 CI 的区别，再说一次
| | pre-commit | CI |
|---|---|---|
| 比喻 | 出门前自查行李 | 机场安检 |
| 在哪跑 | 你的电脑 | GitHub 的全新电脑 |
| 什么时候 | `git commit` 那一刻 | `git push` 之后 |
| 管什么 | **别把不该放的东西放进去**：密码、大文件、格式 | **放进去以后，换一台电脑还能不能跑通、结果还对不对** |
| 能不能跳过 | 能（没装，或者故意跳过） | 不能，结果所有人都看得到 |

---

## 5 · CD：检查通过后自动部署

CI/CD 里的 **CD**，是"检查通过后，自动把新版本部署上线"。公司里：代码合并进主线，CI/CD 自动跑 `terraform apply`、自动更新 Lambda。

**我们项目里**，部署还是手动的：在你电脑上跑 `terraform apply`（01、03、03b 号都是这样）。

---

## 6 · 东西都放在哪：一张总表

| 东西 | 位置 | 进不进 Git |
|---|---|---|
| 程序代码 | `src/` | ✅ |
| 测试 | `tests/` | ✅ |
| 图纸（AWS 上有什么） | `infra/dev/*.tf` | ✅ |
| 数据源清单 | `config/sources.toml` | ✅ |
| notebook | `notebooks/` | ✅（输出会被 nbstripout 清掉） |
| 依赖清单、锁文件 | `pyproject.toml`、`uv.lock` | ✅ |
| pre-commit 的检查清单 | `.pre-commit-config.yaml` | ✅ |
| CI 的检查清单 | `.github/workflows/ci.yml` | ✅ |
| "不许存档"的清单 | `.gitignore` | ✅ |
| Terraform 的记账本 | `infra/dev/terraform.tfstate` | ❌ 可能有敏感信息 |
| 数据 | S3 的 raw 桶 | ❌ 太大，数据在 S3 不在 Git |
| 虚拟环境 | `.venv/` | ❌ 每台电脑用 `uv sync` 自己装 |
| pre-commit 装好的钩子 | `.git/hooks/` | ❌ 每台电脑装一次 |
| 存档历史 | `.git/` | 它就是 Git 本身 |
| CI 用的 AWS 角色名 | GitHub 仓库设置里的变量 `AWS_ROLE_ARN` | ❌ 不在代码里，在 GitHub 的设置里 |

---

## 7 · 一天的正常流程

| 步骤 | 公司里 | 我们 |
|---|---|---|
| 1 | `git pull` 拿到最新的 | 一个人，不用 |
| 2 | 开一个分支 | 直接在 `main` 上 |
| 3 | 改代码，写测试 | 一样 |
| 4 | `git add`、`git commit`（pre-commit 自动自查） | 一样 |
| 5 | `git push`，开 Pull Request，CI 自动跑，同事审查 | `git push` 到 `main`，看 CI 是不是全绿 |
| 6 | 合并，CI/CD 自动部署 | 手动 `terraform apply` |

---

## 8 · 遇到问题怎么办

| 看到什么 | 什么意思 | 怎么做 |
|---|---|---|
| 提交时 pre-commit `Failed`，提示 `files were modified by this hook` | 它自动修好了格式 | 再 `git add -A`，再提交一次 |
| 提交时提示发现私钥或密钥 | 被拦住了，好事 | 把密钥从文件里拿掉，放到环境变量或 Secrets Manager。如果已经推到网上，**马上去 AWS 作废它** |
| `nothing to commit` | 没有新的改动 | 正常，不用管 |
| CI 红了 | 某一项检查没过 | 点进 Actions，看是哪一项、哪一步，日志里有错误。本地修好再推，**红的不合并** |
| 推送被拒：`rejected ... (fetch first)` | GitHub 上有你电脑上没有的新存档 | 先 `git pull`，再推 |
| `Name already exists` | 仓库已经建过了（我们重复运行建仓库那一格时遇到的） | 不用管。看看是不是多出了提交 |

---

## 9 · 看一眼我们自己的仓库

下面几格只看不改。先定义执行命令的小工具：

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)

**全部存档**：每行一次提交，最上面是最新的。

In [ ]:
run("git log --format='%h  %ad  %s' --date=format:'%Y-%m-%d %H:%M'")

**还有没有没存档的改动**：什么都不显示，就是全部存好了。

In [ ]:
run("git status --short")

**`.gitignore` 挡住了哪些东西**：每行显示是 `.gitignore` 里的哪一条规则，挡住了哪个文件。

In [ ]:
run("git check-ignore -v infra/dev/terraform.tfstate infra/dev/build/raw_validator.zip .venv")

**CI 的历史**：每次 push 一行，`success` 是全绿，`failure` 是有红。能看到上面说的两次红，和修好之后的绿。

In [ ]:
run("gh run list --limit 6")

---

## 面试怎么说

1. **What does your development workflow look like?**
   > "Everything is in git. Pre-commit hooks catch secrets, large files and formatting at commit time; every push runs CI on a clean machine — lint, unit and golden tests, Terraform validation, a Docker build and an AWS login check. Nothing merges unless it's green."

2. **Tell me about a time CI caught something.**
   > "Twice in my own project, both invisible locally. First, the workflow pinned a setup-uv version tag that didn't exist — locally I never use that action. Second, GitHub switched to immutable OIDC subject claims that include owner and repository IDs, so AWS rejected the role assumption until I updated the trust condition. Both surfaced on the first CI run instead of in production."

3. **Why have both pre-commit hooks and CI?**
   > "Pre-commit is fast and local, so a secret never leaves the laptop — but it can be skipped. CI can't be skipped and runs on a clean machine, so it proves the project works anywhere."